# Timing and Memory

eXist-db provides pass-through profiling functions that measure execution time and memory without changing your query's result. Wrap any expression to see how long it takes or how much memory it uses.

## util:time — Log Execution Time

[`util:time()`]({docs}/functions/util/time) returns the expression result unchanged and logs the elapsed time. Check your server log for the output.

In [ ]:
util:time(
    for $i in 1 to 10000
    return $i * $i
)

Add a label to identify the measurement in the log:

In [ ]:
util:time(
    for $i in 1 to 10000
    return $i * $i,
    "square numbers"
)

The log will show: `square numbers — 4.2ms`

## util:memory — Log Memory Usage

Same pattern — returns the result unchanged, logs the memory delta:

In [ ]:
util:memory(
    array { 1 to 100000 },
    "large array"
)

## util:track — Structured Measurement

The most useful measurement function. Returns a map with timing, memory, and the actual result:

In [ ]:
let $result := util:track(
    for $i in 1 to 10000
    return $i * $i
)
return map {
    "item-count": count($result?value),
    "time": string($result?time),
    "memory-bytes": $result?memory
}

## Comparing Approaches

Use [`util:track()`]({docs}/functions/util/track) to compare two ways of doing the same thing:

In [ ]:
let $approach-1 := util:track(
    for $i in 1 to 1000
    return string-join(("item", string($i)), "-")
)
let $approach-2 := util:track(
    for $i in 1 to 1000
    return concat("item-", $i)
)
return map {
    "string-join": string($approach-1?time),
    "concat": string($approach-2?time)
}

## With a Label

[`util:track()`]({docs}/functions/util/track) accepts an optional label that appears in the result map:

In [ ]:
util:track(
    for $i in 1 to 5000
    return $i mod 7,
    "modulo computation"
)
=> map:remove("value")

## Nested Timing

Measure individual steps in a pipeline:

In [ ]:
let $step1 := util:track(1 to 10000, "generate")
let $step2 := util:track(
    for $x in $step1?value
    where $x mod 3 = 0
    return $x,
    "filter"
)
let $step3 := util:track(
    sum($step2?value),
    "sum"
)
return map {
    "generate": string($step1?time),
    "filter": string($step2?time),
    "sum": string($step3?time),
    "total-items-generated": count($step1?value),
    "items-after-filter": count($step2?value),
    "final-sum": $step3?value
}

## When to Use Each

| Function | Use When |
|----------|----------|
| [`util:time()`]({docs}/functions/util/time) | Quick check — just want to see timing in the log |
| [`util:memory()`]({docs}/functions/util/memory) | Investigating memory-heavy operations |
| [`util:track()`]({docs}/functions/util/track) | Need the measurement as data (for comparison, reporting) |